# Exploration of the Klinikatlas API

## Retrieve ICD Codes, OPS Codes and basic hospital information

In [38]:
from deutschland import klinikatlas
from deutschland.klinikatlas.api import default_api
import requests

BASE_URL = "https://bundes-klinik-atlas.de"

configuration = klinikatlas.Configuration(
    host=BASE_URL
)

with klinikatlas.ApiClient(configuration) as api_client:
    api = default_api.DefaultApi(api_client)

    icd_codes = api.fileadmin_json_icd_codes_json_get()
    ops_codes = api.fileadmin_json_ops_codes_json_get()
    locations = api.fileadmin_json_locations_json_get()


In [31]:
icd_codes[:5]

[{'description': 'Cholera durch Vibrio cholerae O:1, Biovar cholerae',
  'icdcode': 'A00.0'},
 {'description': 'Cholera durch Vibrio cholerae O:1, Biovar eltor',
  'icdcode': 'A00.1'},
 {'description': 'Cholera, nicht näher bezeichnet', 'icdcode': 'A00.9'},
 {'description': 'Typhus abdominalis und Paratyphus', 'icdcode': 'A01'},
 {'description': 'Typhus abdominalis', 'icdcode': 'A01.0'}]

In [32]:
ops_codes[:5]

[{'description': 'Diagnostik zur Feststellung des irreversiblen '
                 'Hirnfunktionsausfalls',
  'opscode': '1-202'},
 {'description': 'Diagnostik zur Feststellung des irreversiblen '
                 'Hirnfunktionsausfalls: Bei einem potenziellen Organspender',
  'opscode': '1-202.0'},
 {'description': 'Diagnostik zur Feststellung des irreversiblen '
                 'Hirnfunktionsausfalls: Bei einem potenziellen Organspender: '
                 'Ohne Feststellung des irreversiblen Hirnfunktionsausfalls',
  'opscode': '1-202.00'},
 {'description': 'Diagnostik zur Feststellung des irreversiblen '
                 'Hirnfunktionsausfalls: Bei einem potenziellen Organspender: '
                 'Mit Feststellung des irreversiblen Hirnfunktionsausfalls',
  'opscode': '1-202.01'},
 {'description': 'Diagnostik zur Feststellung des irreversiblen '
                 'Hirnfunktionsausfalls: Bei sonstigen Patienten',
  'opscode': '1-202.1'}]

In [33]:
locations[:5]

[{'beds_number': 533,
  'city': 'Rostock',
  'latitude': '54.071629513465',
  'link': 'https://bundes-klinik-atlas.de/krankenhaussuche/krankenhaus/771003/',
  'longitude': '12.107577323914',
  'mail': 'info@kliniksued-rostock.de',
  'name': 'Klinikum Südstadt Rostock',
  'phone': '+49 (0)381/4401-0',
  'street': 'Südring 81',
  'zip': '18059'},
 {'beds_number': 300,
  'city': 'Quedlinburg',
  'latitude': '51.795329141617',
  'link': 'https://bundes-klinik-atlas.de/krankenhaussuche/krankenhaus/771011/',
  'longitude': '11.163533091594',
  'mail': 'info@harzklinikum.com',
  'name': 'Harzklinikum Dorothea Christiane Erxleben GmbH  Standort Quedlinburg',
  'phone': '+49 (0)3946/9090',
  'street': 'Ditfurter Weg 24',
  'zip': '06484'},
 {'beds_number': 262,
  'city': 'Wernigerode',
  'latitude': '51.835429468458',
  'link': 'https://bundes-klinik-atlas.de/krankenhaussuche/krankenhaus/771012/',
  'longitude': '10.774408936550',
  'mail': 'info@harzklinikum.com',
  'name': 'Harzklinikum Dorot

## Retrieve ids for hospitals via search criteria

In [37]:
import requests


def search_hospitals(
    icd=None,
    ops=None,
    location=None,
    start=0,
    rows=10,
):
    """Search hospitals using ICD, OPS, and geographic filters.

    Parameters
    ----------
    icd : str, optional
        ICD code used to filter hospitals by diagnosis.
    ops : str, optional
        OPS code used to filter hospitals by procedure or treatment.
    location : str, optional
        Geographic label used to filter hospitals by location.
    start : int, default=0
        Index of the first result to return.
    rows : int, default=10
        Maximum number of results to return.

    Returns
    -------
    dict
        JSON response from the Bundes-Klinik-Atlas search API.
    """
    params = {
        "tx_solr_start": start,
        "tx_solr_rows": rows,
    }

    if icd is not None:
        params["tx_solr_icd"] = icd

    if ops is not None:
        params["tx_solr_ops"] = ops

    if location is not None:
        params["tx_solr_geolabel"] = location

    response = requests.get(
        f"{configuration.host}/searchresults/",
        params=params,
    )

    response.raise_for_status()

    return response.json()

In [44]:
exemplary_results = search_hospitals(
    icd="A00.0",
    rows=10,
    location="Hamburg"
)

exemplary_results["results"][:2]

[{'id': 773675,
  'header': 'Universitätsklinikum Hamburg-Eppendorf (UKE)',
  'address': 'Martinistraße 52, 20251 Hamburg',
  'detailLink': '/krankenhaussuche/krankenhaus/773675/?tx_tverzhospitaldata_show%5Bquantile%5D=2022%2C5764%2C10079%2C17189&cHash=1db7ecab672bad753b2e00b73e637f7e',
  'content': {'items': [{'header': 'Behandlungsfälle',
     'icon': 'icon-behandlungsfaelle',
     'tooltip': 'Hier sehen Sie, wie viele Patientinnen und Patienten innerhalb eines Jahres in diesem Krankenhaus behandelt wurden und ob das vergleichsweise viel oder wenig ist. Behandlungen psychischer Erkrankungen werden nicht abgebildet.<br><br>Weitere Informationen erhalten Sie <a href="/hilfe-informationen/#c491">hier</a>.',
     'infoData': [{'template': '\\@ce-info-table-short',
       'items': [{'template': '\\@c-tacho-text',
         'data': {'tachoData': {'type5': True, 'scale': 5},
          'text': '<strong>75.856 </strong>(sehr viele)',
          'tooltipText': ''}}]}]},
    {'header': 'Pflegeper

## Find hospital via ID

Searching for an ID returns a HTML page. In this notebook, the HTML page is parsed in a very simple preliminary way to return a JSON with some information for the hospital. We can add more information later.

In [ ]:
from bs4 import BeautifulSoup


def parse_hospital_html(html: str) -> dict:
    """Parse a Bundes-Klinik-Atlas hospital HTML page which
    we get as a result for an id-specific search.

    Parameters
    ----------
    html : str
        HTML returned by `krankenhaussuche_krankenhaus_id_get`.

    Returns
    -------
    dict
        Structured hospital information.
    """

    soup = BeautifulSoup(html, "html.parser")


    title = soup.find("title")
    name = title.get_text(strip=True).split(" | ")[0] if title else None
    def get_text(selector):
        element = soup.select_one(selector)
        return element.get_text(" ", strip=True) if element else None

    def find_text(label):
        """Find text associated with a label."""
        element = soup.find(
            string=lambda text: text and label.lower() in text.lower()
        )

        if element:
            parent = element.parent
            return parent.get_text(" ", strip=True)

        return None

    hospital = {
        "name": name,
        "address": None,
        "phone": None,
        "email": None,
        "website": None,
        "beds": None,
        "treatment_cases": None,
        "departments": [],
    }

    address = soup.find("address")

    if address:
        hospital["address"] = address.get_text(" ", strip=True)

    phone = soup.select_one('a[href^="tel:"]')

    if phone:
        hospital["phone"] = phone.get_text(" ", strip=True)

    email = soup.select_one('a[href^="mailto:"]')

    if email:
        hospital["email"] = email.get_text(" ", strip=True)

    website = soup.select_one(
        'a[href^="http"]:not([href*="bundes-klinik-atlas.de"])'
    )

    if website:
        hospital["website"] = website.get("href")

    for heading in soup.find_all(["h2", "h3", "h4"]):

        text = heading.get_text(" ", strip=True)

        if "Abteilung" in text or "Fachabteilung" in text:
            hospital["departments"].append({
                "name": text
            })

    return hospital

In [ ]:
def get_hospital(hospital_id):
    """Get the detail page of a specific hospital.

    Parameters
    ----------
    hospital_id : int
        Unique ID of the hospital.

    Returns
    -------
    dict
        content of the hospital's detail page.
    """
    with klinikatlas.ApiClient(configuration) as api_client:
        api = default_api.DefaultApi(api_client)

        html = api.krankenhaussuche_krankenhaus_id_get(
            id=hospital_id
        )

    return parse_hospital_html(html)


In [54]:
exemplary_hospital = get_hospital(773675)

exemplary_hospital


{'name': 'Universitätsklinikum Hamburg-Eppendorf (UKE)',
 'address': 'Martinistraße 52, 20251 Hamburg',
 'phone': '+49 (0)40/7410-0',
 'email': 'info@uke.de',
 'website': 'http://www.uke.de',
 'beds': None,
 'treatment_cases': None,
 'departments': [{'name': 'Fachabteilungen'}]}